# Stellarator Island Chain Control

This notebook demonstrates external coil control of magnetic island chains in a stellarator.

## Scientific Background

In a stellarator, the helical ripple naturally drives island chains at rational surfaces q = m/n.
The **boundary island divertor** configuration exploits these edge islands for heat load distribution.
External coils can:

1. **Suppress** an island chain (destructive interference in psi_mn)
2. **Phase-shift** an island chain (rotate island O-points)
3. Create **side effects** -- the press-down-gourd problem

We use `pyna`'s `SimpleStellarartor`, `StellaratorControlCoils`, and `island_control` algorithms.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from pathlib import Path

from pyna.MCF.equilibrium.stellarator import SimpleStellarartor, simple_stellarator
from pyna.MCF.coils.coil_system import StellaratorControlCoils, CoilSet, Biot_Savart_field
from pyna.MCF.control.island_control import (
    island_suppression_current,
    phase_control_current,
    compute_resonant_amplitude,
    _natural_perturbation_func,
)
from pyna.topo.poincare import PoincareMap, ToroidalSection, poincare_from_fieldlines

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'figure.dpi': 150,
    'text.usetex': False,
    'axes.linewidth': 0.8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('pyna loaded successfully')


## 1. Build a SimpleStellarartor with natural island chains

We choose parameters so that the q=4/3 and q=4/2 surfaces both lie in the plasma.

In [ ]:
# q profile: q0=1.1 to q1=5.0  ->  q=4/3~1.33 is near axis, q=4/2=2 at mid-radius
stella = simple_stellarator(
    R0=3.0,
    r0=0.30,
    B0=1.0,
    q0=1.1,
    q1=5.0,
    m_h=4,
    n_h=3,   # ripple drives (4,3) island chain
    epsilon_h=0.05,  # 5% helical ripple
)
print(stella)

# Find resonant surfaces
for m, n in [(4, 4), (4, 3), (4, 2), (3, 2), (5, 4)]:
    psi_list = stella.resonant_psi(m, n)
    if psi_list:
        print(f'  q={m}/{n}={m/n:.3f} -> psi_res={psi_list[0]:.3f}')
    else:
        print(f'  q={m}/{n}={m/n:.3f} -> not in [0,1]')


## 2. Poincare Map: Natural Island Chain (Boundary Island Divertor)

In [ ]:
# Target island: q = 4/3 (exists in the plasma)
TARGET_M, TARGET_N = 4, 3

N_TRANSITS = 60

# Radial scan: covers whole plasma cross-section
R_starts = np.linspace(stella.R0 + 0.02*stella.r0, stella.R0 + 0.93*stella.r0, 24)
start_pts_radial = np.column_stack([R_starts, np.zeros(len(R_starts)), np.zeros(len(R_starts))])
# Near-resonance detail
start_pts_resonance = stella.start_points_near_resonance(TARGET_M, TARGET_N, n_lines=12, delta_psi=0.06)
start_pts = np.vstack([start_pts_radial, start_pts_resonance])

print(f'Tracing {len(start_pts)} field lines near q={TARGET_M}/{TARGET_N}...')

section = ToroidalSection(0.0)
t_max = N_TRANSITS * 2 * np.pi * stella.R0

pmap_natural = poincare_from_fieldlines(
    stella.field_func,
    start_pts,
    sections=[section],
    t_max=t_max,
    dt=0.04,
)
results_natural = pmap_natural.crossing_array(0)  # shape (N, 3): R, Z, phi
print(f'Done. {len(results_natural)} crossings recorded.')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_facecolor('white')

# Color scatter by psi_norm
if len(results_natural) > 0:
    R_pts, Z_pts = results_natural[:, 0], results_natural[:, 1]
    psi_pts = ((R_pts - stella.R0)**2 + Z_pts**2) / stella.r0**2
    psi_norm = np.clip(psi_pts, 0, 1.0)
    cmap = cm.get_cmap('plasma')
    colors = cmap(psi_norm * 0.85 + 0.05)
    ax.scatter(R_pts, Z_pts, s=0.8, c=colors, rasterized=True, alpha=0.7, zorder=2)

# Resonant surface circle
psi_res_target = stella.resonant_psi(TARGET_M, TARGET_N)[0]
r_res = np.sqrt(psi_res_target) * stella.r0
theta_circ = np.linspace(0, 2*np.pi, 300)
ax.plot(stella.R0 + r_res*np.cos(theta_circ), r_res*np.sin(theta_circ),
        '--', color='tomato', lw=0.8, alpha=0.7, label=f'q={TARGET_M}/{TARGET_N} surface')

# O/X points at phi=0 plane
hw = 0.05 * stella.r0
for k in range(TARGET_M):
    theta_O = 2*np.pi*k/TARGET_M
    theta_X = 2*np.pi*k/TARGET_M + np.pi/TARGET_M
    R_O = stella.R0 + r_res * np.cos(theta_O)
    Z_O = r_res * np.sin(theta_O)
    R_X = stella.R0 + r_res * np.cos(theta_X)
    Z_X = r_res * np.sin(theta_X)
    ax.plot([stella.R0+(r_res-hw)*np.cos(theta_O), stella.R0+(r_res+hw)*np.cos(theta_O)],
            [(r_res-hw)*np.sin(theta_O), (r_res+hw)*np.sin(theta_O)],
            '-', color='tomato', lw=3, alpha=0.85, solid_capstyle='round', zorder=5)
    ax.plot(R_O, Z_O, 'o', color='tomato', ms=6, zorder=6, label='O-pt' if k==0 else '')
    ax.plot(R_X, Z_X, 'x', color='tomato', ms=7, mew=1.8, zorder=6, label='X-pt' if k==0 else '')

# LCFS
ax.plot(stella.R0 + stella.r0*np.cos(theta_circ), stella.r0*np.sin(theta_circ),
        'k-', lw=1.2, label='LCFS')

# Colorbar
sm = plt.cm.ScalarMappable(cmap='plasma', norm=Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'$\psi_\mathrm{norm}$', fontsize=11)

ax.set_xlim(stella.R0 - 1.15*stella.r0, stella.R0 + 1.15*stella.r0)
ax.set_ylim(-1.15*stella.r0, 1.15*stella.r0)
ax.set_xlabel('$R$ (m)', fontsize=11)
ax.set_ylabel('$Z$ (m)', fontsize=11)
ax.set_title(f'Natural Poincar\u00e9 map \u2014 $q={TARGET_M}/{TARGET_N}$ island chain\n'
             f'(stellarator, $\\phi=0$)', fontsize=12)
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig('natural_island_poincare.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === Manifold Visualization ===
from pyna.topo.variational import PoincareMapVariationalEquations, _fd_jacobian
from pyna.topo.manifold_improve import StableManifold, UnstableManifold
from pyna.MCF.visual.tokamak_manifold import _manifold_line_collection, manifold_legend_handles
from matplotlib.colors import Normalize
import matplotlib.cm as cm


# field_func_2d: (R, Z, phi) -> [dR/dphi, dZ/dphi]
def field_func_2d(R, Z, phi):
    tang = stella.field_func(np.array([R, Z, phi]))
    dphi_ds = tang[2]
    if abs(dphi_ds) < 1e-15:
        return np.array([0.0, 0.0])
    return np.array([tang[0]/dphi_ds, tang[1]/dphi_ds])


# ------------------------------------------------------------------
# Newton refinement: find the true fixed point of the n-turn map
# ------------------------------------------------------------------
def refine_xpoint(x0, field_func_2d, phi_span, n_iter=12, tol=1e-11):
    """Newton iteration to pin the X-point to machine precision."""
    from scipy.integrate import solve_ivp

    x = np.asarray(x0, dtype=float)
    for i in range(n_iter):
        y0 = np.concatenate([x, np.eye(2).flatten()])

        def aug_rhs(phi, state):
            rz = state[:2]
            M = state[2:].reshape(2, 2)
            f = np.asarray(field_func_2d(rz[0], rz[1], phi), dtype=float)
            A = _fd_jacobian(field_func_2d, rz, phi, eps=1e-7)
            return np.concatenate([f, (A @ M).flatten()])

        sol = solve_ivp(aug_rhs, phi_span, y0,
                        method='DOP853', rtol=1e-11, atol=1e-13, dense_output=False)
        if not sol.success:
            raise RuntimeError(f'refine_xpoint: integrator failed at iter {i}: {sol.message}')

        x_end = sol.y[:2, -1]
        Jac = sol.y[2:, -1].reshape(2, 2)
        residual = x_end - x
        print(f'  Newton iter {i}: |residual|={np.linalg.norm(residual):.2e}  det(J)={np.linalg.det(Jac):.8f}')
        if np.linalg.norm(residual) < tol:
            break
        try:
            delta = np.linalg.solve(Jac - np.eye(2), -residual)
        except np.linalg.LinAlgError:
            break
        x = x + delta
    return x, Jac


# ------------------------------------------------------------------
# Seed candidates: q=m/n island has TARGET_M X-points at phi=0,
# spaced 2*pi/TARGET_M apart in poloidal angle.
# ------------------------------------------------------------------
phi_span = (0.0, 2.0 * np.pi * TARGET_N)   # TARGET_N toroidal turns = one period

psi_res_target = stella.resonant_psi(TARGET_M, TARGET_N)[0]
r_res = np.sqrt(psi_res_target) * stella.r0
import math
_theta_x = math.atan2(-0.064828, 3.056828 - 3.0)

# FIX: q=m/n has m X-points, angular spacing 2*pi/TARGET_M (not /TARGET_N)
_xpt_candidates = [
    (stella.R0 + r_res * np.cos(_theta_x + k * 2*np.pi / TARGET_M),
     r_res * np.sin(_theta_x + k * 2*np.pi / TARGET_M))
    for k in range(TARGET_M)
]

# ------------------------------------------------------------------
# Refine each candidate with Newton iteration; keep the hyperbolic one
# ------------------------------------------------------------------
xpt_RZ, xpt_Jac = None, None
for R_c, Z_c in _xpt_candidates:
    print(f'Trying seed R={R_c:.5f}  Z={Z_c:.5f}')
    try:
        x_ref, M_ref = refine_xpoint([R_c, Z_c], field_func_2d, phi_span)
    except Exception as e:
        print(f'  -> refinement failed: {e}')
        continue
    lam_abs = sorted(np.abs(np.linalg.eigvals(M_ref)))
    det_err = abs(np.linalg.det(M_ref) - 1.0)
    print(f'  -> R={x_ref[0]:.7f}  Z={x_ref[1]:.7f}  lambda_u={lam_abs[1]:.4f}  det_err={det_err:.2e}')
    if lam_abs[1] > 5.0 and det_err < 1e-4:
        xpt_RZ, xpt_Jac = x_ref, M_ref
        print(f'  -> SELECTED as X-point')
        break

if xpt_RZ is None:
    raise RuntimeError(
        'No hyperbolic X-point found with det(J) near 1. '
        'Check field_func_2d or TARGET_M/TARGET_N.'
    )

# ------------------------------------------------------------------
# Grow manifolds with consistent (tight) tolerances
# ------------------------------------------------------------------
RZlimit = (stella.R0 - stella.r0*1.05, stella.R0 + stella.r0*1.05,
           -stella.r0*1.05, stella.r0*1.05)

_ivp_kw = dict(rtol=1e-9, atol=1e-12)   # must match Jacobian integration quality

sm_mfld = StableManifold(xpt_RZ, xpt_Jac, field_func_2d, phi_span=phi_span)
um_mfld = UnstableManifold(xpt_RZ, xpt_Jac, field_func_2d, phi_span=phi_span)

sm_mfld.grow(n_turns=5, init_length=1e-4, n_init_pts=4, both_sides=True, RZlimit=RZlimit, **_ivp_kw)
um_mfld.grow(n_turns=5, init_length=1e-4, n_init_pts=4, both_sides=True, RZlimit=RZlimit, **_ivp_kw)

print(f'Stable   manifold: {len(sm_mfld.segments)} segments')
print(f'Unstable manifold: {len(um_mfld.segments)} segments')

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 6))

rz = pmap_natural[section]
psi_vals = ((rz[:, 0] - stella.R0)**2 + rz[:, 1]**2) / stella.r0**2
sc = ax.scatter(rz[:, 0], rz[:, 1], c=psi_vals, s=0.6, cmap='plasma',
                vmin=0, vmax=1, rasterized=True, alpha=0.55, zorder=2)

s_ref_s = max((seg[:, 0].ptp() for seg in sm_mfld.segments if len(seg) > 1), default=1.0)
for seg in sm_mfld.segments:
    if len(seg) >= 2:
        lc, _ = _manifold_line_collection(seg, cmap='GnBu', s_ref=s_ref_s, lw=1.2)
        ax.add_collection(lc)

s_ref_u = max((seg[:, 0].ptp() for seg in um_mfld.segments if len(seg) > 1), default=1.0)
for seg in um_mfld.segments:
    if len(seg) >= 2:
        lc, _ = _manifold_line_collection(seg, cmap='Oranges', s_ref=s_ref_u, lw=1.2)
        ax.add_collection(lc)

ax.plot(*xpt_RZ, 'kx', ms=10, mew=2, zorder=10, label='X-point')

ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title(f'Poincar\u00e9 + $W^s$/$W^u$ Manifolds, $q={TARGET_M}/{TARGET_N}$ X-point, $\\phi=0$', fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('poincare_manifolds.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Create External Control Coils

In [ ]:
N_COILS = 16    # number of saddle coils
R_COIL = 0.38   # slightly outside plasma (r0=0.30 m)
I0_COIL = 500.0  # reference current (A)

control_coils = StellaratorControlCoils(
    R0=stella.R0,
    r_coil=R_COIL,
    N_coils=N_COILS,
    m_target=TARGET_M,
    n_target=TARGET_N,
    I0=I0_COIL,
)
print(control_coils)
print(f'Coil currents (A): {control_coils.get_currents().round(1)}')


## 4. Island Suppression: Scan Coil Current

In [ ]:
psi_res_target = stella.resonant_psi(TARGET_M, TARGET_N)[0]
nat_func = _natural_perturbation_func(stella)

# Natural amplitude at target surface
b_nat = compute_resonant_amplitude(nat_func, psi_res_target, TARGET_M, TARGET_N, stella,
                                    n_theta=24, n_phi=24)
print(f'Natural |b_tilde_{TARGET_M}{TARGET_N}| = {abs(b_nat):.3e}')

print('Finding island suppression currents...')
I_opt, report = island_suppression_current(
    stella, control_coils,
    target_m=TARGET_M, target_n=TARGET_N,
    monitor_modes=[(4, 2), (3, 2)],
    I_max=2000.0,
    n_theta=24, n_phi=24,
)

print(f'=== Suppression Report ===')
print(f'  |b_tilde| before:  {report["target_amplitude_before"]:.3e}')
print(f'  |b_tilde| after:   {report["target_amplitude_after"]:.3e}')
print(f'  Suppression: {(1 - report["suppression_ratio"])*100:.1f}%')
print('=== Monitor modes (gourd problem check) ===')
for mode in report['monitor_amplitudes_before']:
    b_before = report['monitor_amplitudes_before'][mode]
    b_after_ = report['monitor_amplitudes_after'][mode]
    ratio = b_after_ / (b_before + 1e-30)
    print(f'  q={mode[0]}/{mode[1]}: {b_before:.3e} -> {b_after_:.3e}  (x{ratio:.2f})')


In [ ]:
# Build a helper that adds coil field at unit current
class _UnitCoilSet:
    def __init__(self, coils, I0):
        self.coils = [(pts, I / I0) for pts, I in coils]

unit_coils = _UnitCoilSet(control_coils.coils, I0_COIL)

def coil_field_func(R, Z, phi):
    R, Z, phi = float(R), float(Z), float(phi)
    R_arr = np.array([[R]]); Z_arr = np.array([[Z]]); phi_arr = np.array([[phi]])
    br = bz = bp = 0.0
    for pts, I in unit_coils.coils:
        _br, _bz, _bp = Biot_Savart_field(pts, I, R_arr, Z_arr, phi_arr)
        br += float(_br); bz += float(_bz); bp += float(_bp)
    return br, bz, bp

b_coil_unit = compute_resonant_amplitude(
    coil_field_func, psi_res_target, TARGET_M, TARGET_N, stella, n_theta=20, n_phi=20
)

I0_scan = np.linspace(0, 1500, 12)
b_total_scan = [abs(b_nat + b_coil_unit * I0) for I0 in I0_scan]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(I0_scan, np.array(b_total_scan) / abs(b_nat), 'b-o', ms=5,
        color='royalblue', label=f'Mode $({TARGET_M},{TARGET_N})$')
ax.axhline(1, color='gray', ls=':', lw=0.8, label='Natural level')
ax.axhline(0, color='tomato', ls='--', lw=0.8, alpha=0.6, label='Full suppression')
ax.set_xlabel('Control coil current $I_0$ (A)', fontsize=11)
ax.set_ylabel(r'$|\tilde{b}_{mn}| / |\tilde{b}_{mn}^{\rm nat}|$', fontsize=11)
ax.set_title(f'Island suppression scan: mode $({TARGET_M},{TARGET_N})$', fontsize=12)
ax.grid(True, alpha=0.3, lw=0.5)
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig('island_suppression_scan.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Press-Down-Gourd Problem

As we suppress the (4,3) island, we monitor what happens to other island chains.

In [ ]:
MONITOR_M, MONITOR_N = 4, 2
psi_res_monitor = stella.resonant_psi(MONITOR_M, MONITOR_N)

if psi_res_monitor:
    psi_res_mon = psi_res_monitor[0]
    b_nat_mon = compute_resonant_amplitude(
        nat_func, psi_res_mon, MONITOR_M, MONITOR_N, stella, n_theta=20, n_phi=20)
    b_coil_mon = compute_resonant_amplitude(
        coil_field_func, psi_res_mon, MONITOR_M, MONITOR_N, stella, n_theta=20, n_phi=20
    )
    b_target_scan = [abs(b_nat + b_coil_unit * I0) for I0 in I0_scan]
    b_monitor_scan = [abs(b_nat_mon + b_coil_mon * I0) for I0 in I0_scan]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    ax1.plot(I0_scan, np.array(b_target_scan)/abs(b_nat), '-o', ms=5,
             color='royalblue', label=f'Target $({TARGET_M},{TARGET_N})$')
    ax1.axhline(1, color='gray', ls=':', lw=0.8)
    ax1.set_ylabel(r'Norm. $|\tilde{b}_{mn}|$', fontsize=11)
    ax1.legend(fontsize=9, framealpha=0.9)
    ax1.grid(True, alpha=0.3, lw=0.5)
    ax1.set_title('Press-Down-Gourd: Island control side effects', fontsize=12)

    ax2.plot(I0_scan, np.array(b_monitor_scan)/abs(b_nat_mon), '-s', ms=5,
             color='tomato', label=f'Monitor $({MONITOR_M},{MONITOR_N})$')
    ax2.axhline(1, color='gray', ls=':', lw=0.8)
    ax2.set_xlabel('Control coil current $I_0$ (A)', fontsize=11)
    ax2.set_ylabel(r'Norm. $|\tilde{b}_{mn}|$', fontsize=11)
    ax2.legend(fontsize=9, framealpha=0.9)
    ax2.grid(True, alpha=0.3, lw=0.5)

    plt.tight_layout()
    plt.savefig('gourd_problem.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Monitor surface q={MONITOR_M}/{MONITOR_N} not in plasma')


## 6. Phase Control: Rotating Island O-Points

In [ ]:
# Phase control: vary desired island O-point phase
# Show how island amplitude responds when coils are phase-shifted
phase_shifts = np.linspace(0, 2*np.pi, 17)
b_phase = []

for dphase in phase_shifts:
    cc_p = StellaratorControlCoils(
        R0=stella.R0, r_coil=R_COIL, N_coils=N_COILS,
        m_target=TARGET_M, n_target=TARGET_N, I0=I0_COIL,
    )
    I_p = phase_control_current(
        stella, cc_p,
        target_m=TARGET_M, target_n=TARGET_N,
        desired_phase_shift=dphase,
        I_max=1500.0, n_theta=20, n_phi=20,
    )

    # Build coil_field_func for THIS phase-shifted coil set with optimized currents
    class _PhaseCoilSet:
        def __init__(self, coils, currents):
            self.pairs = list(zip(coils, currents))
    pcoils = _PhaseCoilSet(cc_p.coils, I_p)

    def phase_coil_field_func(R, Z, phi, _pairs=pcoils.pairs):
        R, Z, phi = float(R), float(Z), float(phi)
        R_arr = np.array([[R]]); Z_arr = np.array([[Z]]); phi_arr = np.array([[phi]])
        br = bz = bp_val = 0.0
        for pts, I in _pairs:
            _br, _bz, _bp = Biot_Savart_field(pts, float(I), R_arr, Z_arr, phi_arr)
            br += float(_br); bz += float(_bz); bp_val += float(_bp)
        return np.array([br, bz, bp_val])

    b_p = compute_resonant_amplitude(
        phase_coil_field_func, psi_res_target, TARGET_M, TARGET_N, stella,
        n_theta=16, n_phi=16
    )
    b_phase.append(abs(b_nat + b_p))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.degrees(phase_shifts), np.array(b_phase)/abs(b_nat), '-o', ms=5,
        color='seagreen', label=f'Mode $({TARGET_M},{TARGET_N})$')
ax.axhline(1, color='gray', ls=':', lw=0.8, label='Natural level')
ax.set_xlabel('Desired phase shift (degrees)', fontsize=11)
ax.set_ylabel(r'$|\tilde{b}_{mn}| / |\tilde{b}_{mn}^{\rm nat}|$', fontsize=11)
ax.set_title(f'Phase control: mode $({TARGET_M},{TARGET_N})$ response vs. phase shift', fontsize=12)
ax.grid(True, alpha=0.3, lw=0.5)
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig('phase_control.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Export Poincare Data as JSON

In [ ]:
import json as _json, os
os.makedirs('pyna_output', exist_ok=True)

poincare_data = {
    'phi_section': 0.0,
    'target_mode': [TARGET_M, TARGET_N],
    'R': results_natural[:, 0].tolist() if len(results_natural) > 0 else [],
    'Z': results_natural[:, 1].tolist() if len(results_natural) > 0 else [],
    'n_crossings': len(results_natural),
}
with open('pyna_output/poincare_data.json', 'w') as f:
    _json.dump(poincare_data, f, indent=2)
print(f'Exported poincare_data.json ({poincare_data["n_crossings"]} crossings)')

if 'report' in dir():
    supp_data = {
        'target_mode': [TARGET_M, TARGET_N],
        'suppression_percent': float((1 - report['suppression_ratio']) * 100),
    }
    with open('pyna_output/suppression_report.json', 'w') as f:
        _json.dump(supp_data, f, indent=2)
    print(f'Exported suppression_report.json')
    print(f'  Suppression: {supp_data["suppression_percent"]:.1f}%')


## 9. Multi-Objective Island Optimisation

`IslandOptimizer` minimises a weighted sum of:
1. Resonant amplitude $|\\tilde{b}_{mn}|^2$ \u2192 suppress internal islands  
2. X-point eigenvalue deviation $(|\\lambda_u|-1)^2$ \u2192 control boundary island  
3. Monitor-mode penalty (don't amplify other chains)  
4. Chirikov overlap constraint $\\sigma \\le \\sigma_{\\max}$  

The coil response matrix $R_{mn}^{(k)} = \\partial\\tilde{b}_{mn}/\\partial I_k$ is built  
once by unit-current sweeps and cached.

In [ ]:
from pyna.MCF.control.island_optimizer import (
    IslandOptimizer,
    UnperturbedSurfaceReconstructor,
    compute_surface_deformation,
    epsilon_eff_proxy,
    _make_coil_field_func,
)

# Re-use the same coils built in Section 3, reset to zero
coils.set_amplitude(0.0)

opt = IslandOptimizer(
    stella,
    coils,
    target_suppress=[(4, 3)],
    target_boundary=[(2, 1)],
    monitor_modes=[(3, 1)],
    w_suppress=2.0,
    w_boundary=0.5,
    w_monitor=1.0,
    sigma_max=0.85,
    phi0=0.0,
    n_theta=16,
    n_phi=16,
)

all_modes = [(4,3),(2,1),(3,1)]
opt._build_response(all_modes, verbose=True)
print(f"Response matrix condition number: {opt.condition_number():.3e}")

In [ ]:
result = opt.optimise(
    I_max=5e3,
    method='L-BFGS-B',
    include_eigenvalue=False,   # skip for CI speed; set True for research use
    n_restarts=2,
    verbose=True,
)
result.summary()

### Pareto Front: Island Suppression vs. Boundary Eigenvalue

In [ ]:
pareto = opt.pareto_scan(I_max=5e3, n_weights=5, verbose=False)
result.pareto_front = pareto

fig, ax = plt.subplots(figsize=(5, 4))
objs = np.array([p[1] for p in pareto])
ax.plot(objs[:, 0], objs[:, 1], 'o-', color='steelblue', ms=7)
ax.set_xlabel(r'Island suppression objective  $\Sigma|\tilde{b}_{mn}|^2$')
ax.set_ylabel(r'Boundary eigenvalue objective  $\Sigma(|\lambda_u|-1)^2$')
ax.set_title('Pareto front')
plt.tight_layout()
plt.show()

### Side Effects: Non-Resonant Deformation & Neoclassical Transport

In [ ]:
# Apply optimal currents
coils.set_currents(result.currents)
coil_func_opt = _make_coil_field_func(coils)

# Neoclassical transport proxy
S_check = np.linspace(0.15, 0.85, 8)
eps_before = epsilon_eff_proxy(stella, None, S_check)
eps_after  = epsilon_eff_proxy(stella, coil_func_opt, S_check, coil_currents=result.currents)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(S_check, eps_before, 'b-', lw=2, label='Before')
axes[0].plot(S_check, eps_after,  'r--', lw=2, label='After')
axes[0].set_xlabel('S'); axes[0].set_ylabel(r'$\varepsilon_{\rm eff}$ proxy')
axes[0].set_title('Neoclassical transport proxy'); axes[0].legend()

# Before/after Poincare comparison
def field_func_total(rzphi):
    tang = stella.field_func(rzphi)
    dphi_ds = tang[2]
    if abs(dphi_ds) < 1e-15:
        return tang
    try:
        R, Z, phi = rzphi[0], rzphi[1], rzphi[2]
        br_c, bz_c, _ = coil_func_opt(R, Z, phi)
        B_phi_bg = stella.B0 * stella.R0 / R
        scale = 1.0 / (B_phi_bg / R + 1e-30)
        dRdphi = tang[0]/dphi_ds + br_c*scale
        dZdphi = tang[1]/dphi_ds + bz_c*scale
        norm = np.sqrt(dRdphi**2 + dZdphi**2 + 1.0)
        return np.array([dRdphi/norm, dZdphi/norm, 1.0/norm])
    except Exception:
        return tang

from pyna.topo.poincare import poincare_from_fieldlines as _pfl
sec_after = _pfl(field_func_total, start_pts, n_turns=60,
                 phi_target=0.0, method='DOP853', rtol=1e-8, atol=1e-11)

for pts in results_natural[0]:   # 'before' from section 2
    if len(pts) > 1:
        axes[1].plot(pts[:, 0], pts[:, 1], ',', ms=0.8, alpha=0.4, color='steelblue')
for pts in sec_after:
    if len(pts) > 1:
        axes[1].plot(pts[:, 0], pts[:, 1], ',', ms=0.8, alpha=0.4, color='darkorange')
axes[1].set_aspect('equal')
axes[1].set_xlabel('R (m)'); axes[1].set_ylabel('Z (m)')
axes[1].set_title('Poincare: before (blue) / after (orange)')

plt.tight_layout()
plt.savefig('poincare_before_after_optimizer.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Suppression (4,3): {result.suppression_before[(4,3)]:.3e} -> {result.suppression_after[(4,3)]:.3e}")
print(f"Transport change: {result.transport_change:+.3f}")

In [ ]:
# --- (m,n) Fourier spectrum heatmap: before and after control ---
from pyna.MCF.visual.RMP_spectrum import compute_mn_spectrum, plot_mn_heatmap
from pyna.MCF.control.island_optimizer import _make_coil_field_func
from pyna.MCF.control.island_control import _natural_perturbation_func

# Natural (background) field perturbation function
nat_func_bg = _natural_perturbation_func(stella)

# Coil field at optimal currents
coil_func_after = _make_coil_field_func(coils)  # coils already set to optimal

def total_pert_func(R, Z, phi):
    """Natural + coil perturbation."""
    db_nat = nat_func_bg(R, Z, phi)
    try:
        db_coil = coil_func_after(R, Z, phi)
        return [db_nat[0] + db_coil[0], db_nat[1] + db_coil[1], 0.0]
    except Exception:
        return db_nat

# Compute spectra at q=4/3 resonant surface
S_res = stella.resonant_psi(TARGET_M, TARGET_N)[0]
M_MAX, N_MAX = 5, 3

b_mn_nat   = compute_mn_spectrum(nat_func_bg,   S_res, stella, m_max=M_MAX, n_max=N_MAX, n_theta=24, n_phi=24)
b_mn_total = compute_mn_spectrum(total_pert_func, S_res, stella, m_max=M_MAX, n_max=N_MAX, n_theta=24, n_phi=24)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_mn_heatmap(b_mn_nat, m_max=M_MAX, n_max=N_MAX, ax=axes[0],
                log_scale=True, cmap='hot_r',
                title=r'$|\tilde{b}_{mn}|$ \u2014 Natural field  (S=' + f'{S_res:.3f})',
                highlight_modes=[(TARGET_M, TARGET_N)])

plot_mn_heatmap(b_mn_total, m_max=M_MAX, n_max=N_MAX, ax=axes[1],
                log_scale=True, cmap='hot_r',
                title=r'$|\tilde{b}_{mn}|$ \u2014 After optimisation  (S=' + f'{S_res:.3f}'),
                highlight_modes=[(TARGET_M, TARGET_N)])

fig.suptitle(f'(m,n) Fourier spectrum: target mode ({TARGET_M},{TARGET_N}) highlighted (red box)', fontsize=11)
plt.tight_layout()
plt.savefig('mn_spectrum_before_after.png', dpi=120, bbox_inches='tight')
plt.show()

# Print the change in (4,3) amplitude
amp_nat   = abs(b_mn_nat  [M_MAX + TARGET_M, N_MAX + TARGET_N])
amp_after = abs(b_mn_total[M_MAX + TARGET_M, N_MAX + TARGET_N])
print(f"b_({TARGET_M},{TARGET_N}) before: {amp_nat:.4e}")
print(f"b_({TARGET_M},{TARGET_N}) after:  {amp_after:.4e}")
print(f"Suppression ratio: {amp_after/(amp_nat+1e-30):.4f}")

## 8. Summary

| Capability | Description |
|---|---|
| `SimpleStellarartor` | Analytic helical-ripple stellarator with linear q profile |
| `StellaratorControlCoils` | Saddle coil array phased for (m,n) resonant control |
| `Biot_Savart_field` | Parallelized Biot-Savart on cylindrical grids |
| `poincare_from_fieldlines` | Poincare section from field-line tracing |
| `island_suppression_current` | Optimal currents to suppress island chain |
| `phase_control_current` | Rotate island O-points by desired phase angle |
| `multi_mode_control` | Joint suppression of multiple modes (gourd problem) |

The press-down-gourd problem is visible: suppressing (4,3) can amplify (4,2). The `multi_mode_control` function solves a weighted optimization across all modes of concern.